# Chapter 4: Hamiltonian Construction & Qubit Mapping

## What you'll learn

- What information the classical Hamiltonian encodes and how integral counts scale with active space size
- How to build Hamiltonians for different active spaces and read `get_summary()` diagnostically
- The three qubit encodings available in the QDK: Jordan-Wigner, Bravyi-Kitaev, and Parity
- How to verify that all encodings are equivalent via exact diagonalization
- How the Schatten norm connects your qubit Hamiltonian directly to QPE circuit cost
- How to build model Hamiltonians (Ising, Heisenberg, Hubbard) using `LatticeGraph` for hardware benchmarking

## From active space to qubits

Chapter 3 produced an active space (via autoCAS-EOS) and showed that choosing fewer orbitals saves qubits. This chapter takes the next step: converting that active space into a qubit Hamiltonian that a quantum computer can act on.

The conversion has two stages. First, the classical Hamiltonian is constructed from the one- and two-body integrals of the active orbital basis. Second, a qubit mapper applies a fermion-to-qubit encoding that rewrites fermionic creation and annihilation operators as Pauli strings. Different encodings distribute fermionic anti-commutation relations differently across qubits — they are unitarily equivalent but can differ in circuit depth for specific hardware.

In [ ]:
# Setup: SCF → valence → MP2 → autoCAS-EOS pipeline
from pathlib import Path

import qdk_chemistry.plugins.pyscf
import qdk_chemistry.plugins.qiskit

from qdk_chemistry.data import Structure
from qdk_chemistry.algorithms import available, create, print_settings
from qdk_chemistry.utils import Logger, compute_valence_space_parameters

Logger.set_global_level(Logger.LogLevel.off)

N2_XYZ = Path("../examples/data/stretched_n2.structure.xyz")
structure = Structure.from_xyz_file(N2_XYZ)

E_hf, wfn_hf = create("scf_solver").run(structure, charge=0, spin_multiplicity=1, basis_or_guess="cc-pvdz")
num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
wfn_val = create("active_space_selector", "qdk_valence",
                 num_active_electrons=num_val_e, num_active_orbitals=num_val_o).run(wfn_hf)
val_idx = wfn_val.get_orbitals().get_active_space_indices()
wfn_mp2 = create("orbital_localizer", "qdk_mp2_natural_orbitals").run(wfn_val, *val_idx)

loc_ham = create("hamiltonian_constructor").run(wfn_mp2.get_orbitals())
macis = create("multi_configuration_calculator", "macis_asci",
               calculate_one_rdm=True, calculate_two_rdm=True)
macis.settings().set("core_selection_strategy", "fixed")
n_a, n_b = wfn_mp2.get_active_num_electrons()
_, wfn_sci = macis.run(loc_ham, n_a, n_b)
wfn_eos = create("active_space_selector", "qdk_autocas_eos").run(wfn_sci)

eos_indices = wfn_eos.get_orbitals().get_active_space_indices()[0]
print(f"HF energy: {E_hf:.6f} Hartree")
print(f"autoCAS-EOS active space: {len(eos_indices)} orbitals → {2 * len(eos_indices)} qubits")


## Part 1: Building the active space Hamiltonian

The `hamiltonian_constructor` takes an `Orbitals` object (from any wavefunction) and computes the one- and two-body integrals in that orbital basis. The result is a classical Hamiltonian object that stores everything needed for both exact diagonalization and qubit mapping.

`get_summary()` exposes four diagnostically useful fields: active orbital count (qubit cost), core energy (constant offset folded into the qubit identity term), and the one- and two-body integral counts with a sparsity indicator (how many are above a threshold).

In [ ]:
# Inspect hamiltonian_constructor settings
print("Available Hamiltonian constructors:", available("hamiltonian_constructor"))
print()
print_settings("hamiltonian_constructor", "qdk")


In [ ]:
# Build the active space Hamiltonian and interpret the summary
ham_constructor = create("hamiltonian_constructor")
active_ham = ham_constructor.run(wfn_eos.get_orbitals())
print(active_ham.get_summary())

# Reading the summary:
#   Core Energy      — energy of frozen (non-active) electrons + nuclear repulsion.
#                      This constant is carried into the identity term of the qubit Hamiltonian.
#   One-body Integrals — kinetic + nuclear-attraction matrix elements; n^2 entries for n orbitals.
#   Two-body Integrals — electron-electron repulsion; n^4 entries. This is the dominant scaling term.
#   Active Orbitals  — sets the qubit count: 2 × n_orbitals under Jordan-Wigner or Bravyi-Kitaev.


## Part 2: Active space size and integral scaling

Two-body integrals $h_{pqrs}$ have four orbital indices and scale as $n^4$ in the number of active orbitals. Going from 8 to 4 orbitals reduces the two-body table from 4096 to 256 entries — a 16× reduction. After qubit mapping, this translates directly into fewer Pauli terms and a shallower circuit. Let's build Hamiltonians for the full valence space and the autoCAS-EOS space. Compare the integral statistics in `get_summary()`.

In [ ]:
# Compare Hamiltonians for two active space sizes
ham_constructor = create("hamiltonian_constructor")
valence_ham = ham_constructor.run(wfn_mp2.get_orbitals())   # full valence: 8 orbitals
active_ham  = ham_constructor.run(wfn_eos.get_orbitals())   # autoCAS-EOS:  4 orbitals

print("─── Full valence (8 orbitals, 16 qubits) ───")
print(valence_ham.get_summary())
print("─── autoCAS-EOS (4 orbitals, 8 qubits) ───")
print(active_ham.get_summary())

# Two-body integrals scale as n^4: 8^4 = 4096 vs 4^4 = 256 (16x reduction).
# This directly determines the number of Pauli terms after qubit mapping.
# Halving the active space gives an exponential reduction in circuit complexity downstream.


## Part 3: What qubit mappers are available?

Three fermion-to-qubit encodings are available in the QDK:

**<a href="https://www.tandfonline.com/doi/abs/10.1080/00268976.2011.552441" target="_blank">Jordan-Wigner</a>**: maps each spin-orbital to one qubit; the anti-commutation relation is enforced by Z-string parity chains. Intuitive, but Z-strings grow with system size.

**<a href="https://aip.scitation.org/doi/10.1063/1.4768229" target="_blank">Bravyi-Kitaev</a>**: encodes both occupation and parity in a tree structure; reduces the average Z-string length to O(log n). Often preferred for circuits.

**Parity**: reorganizes the encoding around the parity of occupied orbitals; allows two qubits to be tapered off for particle-number-conserving Hamiltonians. Requires the `qiskit` plugin (`import qdk_chemistry.plugins.qiskit`).

The three encodings are unitarily equivalent — they represent the same operator and produce identical spectra. The broader mapping landscape offers further alternatives; for recent work see <a href="https://journals.aps.org/pra/abstract/10.1103/PhysRevA.100.032337" target="_blank">Steudtner & Wehner (2019)</a> and <a href="https://arxiv.org/pdf/2303.02270" target="_blank">arXiv:2303.02270</a>.

**Of these three, Jordan-Wigner is the only encoding supported end-to-end in the QPE pipeline today.** BK and Parity are fully supported for Hamiltonian construction and exact diagonalization — and verified to agree with JW in Part 5 below — but are not yet wired into the time evolution and circuit execution chain.

Now, run the cell to see both mapper settings.

In [ ]:
# List available qubit mappers and their encoding options
print("Available qubit mappers:", available("qubit_mapper"))
print()
for name in available("qubit_mapper"):
    print(f"--- {name} ---")
    print_settings("qubit_mapper", name)
    print()

# Two mapper implementations: 'qdk' (native) and 'qiskit' (requires plugin).
# 'qdk' supports Jordan-Wigner and Bravyi-Kitaev.
# 'qiskit' additionally supports Parity encoding.
# All encodings preserve the spectrum — they differ in how locality is distributed across qubits.


## Part 4: Applying all encoding schemes

All five encoding variants are applied to the same autoCAS-EOS Hamiltonian. The output table shows qubit count, Pauli term count, and Schatten norm for each.

For this small, highly symmetric system (N₂, 4 orbitals, 8 qubits) the Pauli term counts are identical across encodings. For larger, less symmetric molecules the counts diverge — BK typically reduces terms relative to JW for mid-sized systems.

The **Schatten norm** (sum of |Pauli coefficients|) is encoding-independent: it is a property of the Hamiltonian, not its representation. It can be used to set the maximum useful evolution time in QPE. The Schatten (1-)norm of the qubit Hamiltonian is the sum of the absolute values of all Pauli coefficients. In first-order Trotterized QPE, this norm bounds the maximum useful evolution time: T_max = π / norm (<a href="https://arxiv.org/abs/1912.08854" target="_blank">Childs et al., 2021</a>). A larger norm forces a shorter T_max — and since QPE energy resolution scales as 2π / (T · 2ⁿ), a smaller T means more phase bits or more Trotter repetitions are needed to reach a target precision.

This is the quantitative link between active space selection (Chapter 3) and QPE cost (Chapter 6): a larger active space produces larger Hamiltonian coefficients, a larger norm, a shorter T_max, and therefore a more demanding QPE calculation.

Before filling in the missing encoding name in the cell below, think through how the encoding affects the number of Pauli terms. Answer the question below, and then run the cell.

# Question: ch4-encoding-bhc6bf

> Question not found (HTTP 404)


In [ ]:
# Apply all encoding schemes to the autoCAS-EOS Hamiltonian
encoding_cases = [
    ("qdk",    "jordan-wigner"),
    ("qdk",    "bravyi-kitaev"),
    ("qiskit", "jordan-wigner"),
    ("qiskit", "bravyi-kitaev"),
    ("qiskit", "???"),              # fill in: the third qiskit encoding (not JW or BK)
]

qubit_hams = {}
print(f"{'Mapper / Encoding':<30} {'Qubits':>7} {'Pauli terms':>12} {'Schatten norm':>14}")
print("-" * 67)
for impl, enc in encoding_cases:
    key = f"{impl}/{enc}"
    qham = create("qubit_mapper", impl, encoding=enc).run(active_ham)
    qubit_hams[key] = qham
    print(f"{key:<30} {qham.num_qubits:>7} {len(qham.pauli_strings):>12} {qham.schatten_norm:>14.4f}")

print()
print("For this small, highly symmetric system all encodings produce the same")
print("Pauli term count. For larger, lower-symmetry molecules the counts diverge.")
print("The Schatten norm is encoding-independent — it depends only on the Hamiltonian")
print("coefficients, not their Pauli representation. It sets the QPE evolution time.")

# Question: ch4-integral-scaling-ptrlft

> Question not found (HTTP 404)


## Part 5: Verifying energy agreement and solver comparison

All encodings are unitary-equivalent representations of the same operator: their ground-state eigenvalues must agree exactly. This cell runs exact diagonalization on every encoding and checks agreement, then compares the dense and sparse matrix solvers.

The **dense solver** builds the full 2ⁿ× 2ⁿ Hilbert space matrix explicitly. This is simple but uses O(4ⁿ) memory. For 8 qubits (2⁸ = 256) it is fast; for 16 qubits (2¹⁶ = 65536) it is slow.

The **sparse solver** uses iterative Lanczos/Davidson methods, only storing the non-zero Hamiltonian entries. Much more efficient for larger qubit counts; the recommended default for n > 12 qubits.

In the cell below, choose which solver to use and then confirm all encodings agree. Make sure you replace the "???" with the matrix solver. 

In [ ]:
# Inspect qubit_hamiltonian_solver settings
print("Available qubit Hamiltonian solvers:", available("qubit_hamiltonian_solver"))
print()
for name in available("qubit_hamiltonian_solver"):
    print(f"--- {name} ---")
    print_settings("qubit_hamiltonian_solver", name)
    print()

# Two solvers:
#   qdk_sparse_matrix_solver — iterative Lanczos/Davidson; stores only non-zero entries;
#                              efficient for systems up to ~20 qubits.
#   qdk_dense_matrix_solver  — builds the full 2^n × 2^n matrix in memory;
#                              only practical for very small qubit counts (< 14 qubits).


In [ ]:
# Verify energy agreement across encodings and compare solvers
# Hint: use qdk_sparse_matrix_solver for systems beyond ~12 qubits; dense for smaller ones
solver = create("qubit_hamiltonian_solver", "???")  # choose: "qdk_dense_matrix_solver" or "qdk_sparse_matrix_solver"

print(f"{'Mapper / Encoding':<30} {'Energy (Hartree)':>20}")
print("-" * 52)
for key, qham in qubit_hams.items():
    energy, _ = solver.run(qham)
    print(f"{key:<30} {energy:>20.6f}")

print()
print("All encodings are unitarily equivalent representations of the same operator.")
print("Exact diagonalization is encoding-agnostic: eigenvalues must agree exactly.")
print()
dense_solver = create("qubit_hamiltonian_solver", "qdk_dense_matrix_solver")
e_dense, _ = dense_solver.run(qubit_hams["qdk/jordan-wigner"])
e_sparse, _ = create("qubit_hamiltonian_solver", "qdk_sparse_matrix_solver").run(qubit_hams["qdk/jordan-wigner"])
print(f"Dense  solver: {e_dense:.6f} Hartree")
print(f"Sparse solver: {e_sparse:.6f} Hartree  (← same eigenvalue, more memory-efficient algorithm)")

## Summary

In this chapter you:
- Built classical Hamiltonians for two active spaces and compared how integral counts scale with n⁴
- Listed the three qubit encodings (JW, BK, Parity) and confirmed Jordan-Wigner is the only one supported end-to-end in the QPE pipeline today
- Applied all encoding schemes and verified they produce identical spectra
- Computed the Schatten norm and its connection to the QPE time parameter T_max

**Key pattern:**
```python
ham_constructor = create("hamiltonian_constructor")
active_ham = ham_constructor.run(wfn_eos.get_orbitals())

qubit_mapper = create("qubit_mapper", "qdk", encoding="jordan-wigner")
qubit_ham = qubit_mapper.run(active_ham)

solver = create("qubit_hamiltonian_solver", "qdk_sparse_matrix_solver")
energy, _ = solver.run(qubit_ham)
```

The `active_ham` and `qubit_ham` objects are carried forward into Chapter 5 (state preparation).